# SRC LEARNING · Gradio trên Google Colab

Chạy hai ô theo thứ tự. Ô cuối tạo link Gradio; chạm link đó để mở giao diện trong tab Safari riêng. Dán Google Drive File ID của môn được phê duyệt để tải dữ liệu, rồi dùng Flash Cards, Học cấp tốc và Mock Exam.

**Phiên mới:** Mỗi lần chạy lại ô cuối, ứng dụng tạo DATA và SQLite tạm mới. Dữ liệu và tiến độ mất khi runtime bị xóa; không đồng bộ với Desktop.

**Link chia sẻ:** Ai có link `gradio.live` đều có thể mở app; không đưa link cho người khác nếu bạn đang dùng dữ liệu riêng. Link hết hiệu lực khi runtime hoặc ô Gradio dừng. Colab miễn phí có thể ngắt runtime khi chủ yếu sử dụng web UI ngoài notebook.


In [ ]:
from pathlib import Path
import subprocess, sys, tempfile
REPO_URL = "https://github.com/ScottPhantom/SRC-LEARNING.git"
BRANCH = "Mobile_Web_APP"
SOURCE_DIR = Path(tempfile.mkdtemp(prefix="src-learning-source-")) / "SRC-LEARNING"
subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, REPO_URL, str(SOURCE_DIR)], check=True)
actual_branch = subprocess.check_output(["git", "-C", str(SOURCE_DIR), "branch", "--show-current"], text=True).strip()
assert actual_branch == BRANCH, f"Đã tải nhánh khác: {actual_branch}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(SOURCE_DIR / "requirements-mobile.txt")], check=True)
print("Đã tải nhánh", actual_branch, "commit", subprocess.check_output(["git", "-C", str(SOURCE_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import importlib, os, sys, tempfile
from IPython.display import HTML, display

previous_demo = globals().get("gradio_demo")
if previous_demo is not None:
    close_demo = getattr(previous_demo, "close", None) or getattr(getattr(previous_demo, "blocks", None), "close", None)
    if callable(close_demo):
        close_demo()
previous_session = getattr(globals().get("mobile_app"), "_session", None)
if previous_session is not None:
    previous_session.close()
SESSION_ROOT = tempfile.mkdtemp(prefix="src-learning-session-")
os.environ["SRC_LEARNING_SESSION_ROOT"] = SESSION_ROOT
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
for module_name in tuple(sys.modules):
    if module_name in {"app", "colab_mobile"} or module_name.startswith(("app.", "colab_mobile.")):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()
import colab_mobile.gradio_app as mobile_app
gradio_demo, local_url, share_url = mobile_app.launch_app(share=True, prevent_thread_lock=True)
if not share_url:
    raise RuntimeError("Gradio không tạo được link chia sẻ. Hãy xem log và chạy lại ô này.")
display(HTML(f'<p><a href="{share_url}" target="_blank" rel="noopener noreferrer" style="font-size:20px;font-weight:700">Mở SRC Learning trong tab Safari mới ↗</a></p>'))
print("Link phiên này:", share_url)
